# VISCERA — concept-supervised method (full pipeline, A6000/A100)
**VIS**ual-**C**oncept **E**ndoscopy **R**epresentation via **A**ttributes. Pipeline: SSL weight (dinov2.pth) → **Stage-1: concept-supervised pretrain on ~160k (35 clinical concepts)** with **L2-SP anchor** (keep SSL features) + GRL center-invariance → **Stage-2: downstream neo/ndbe** (PPV@90R tail loss + WiSE-FT + multi-seed ensemble).

**Drive `DRIVE_DIR` needs:** numbered data zips `0.zip..N.zip` (→ out/<dir>), `train.zip`/`val.zip` (→ out/train, out/val), `dinov2.pth`. No dataset.zip needed (labels come from the JSONs). Runtime → **GPU**.

In [ ]:
import torch; print(torch.__version__)
!nvidia-smi --query-gpu=name,memory.total --format=csv
!pip -q install timm==1.0.27 scikit-learn

In [ ]:
REPO_URL = 'https://github.com/HuynhDoTanThanh/RARE2026.git'   # <-- your repo
%cd /content
!rm -rf rare && git clone $REPO_URL rare
%cd /content/rare

In [ ]:
# ---- extract data from Drive (numbered chunks -> out/<dir>; train/val.zip -> out/train,out/val) ----
from google.colab import drive; drive.mount('/content/drive')
DRIVE_DIR = '/content/drive/MyDrive/RARE_LG'   # <-- your Drive folder
import glob, os, zipfile, shutil
assert os.path.exists(f'{DRIVE_DIR}/dinov2.pth'), f'upload dinov2.pth to {DRIVE_DIR}'
shutil.copy(f'{DRIVE_DIR}/dinov2.pth', 'dinov2.pth'); print('dinov2.pth OK')
os.makedirs('out', exist_ok=True)
def extract_chunk(zpath):
    with zipfile.ZipFile(zpath) as zf:
        tops = {p.split('/')[0] for p in zf.namelist() if p.strip('/')}
        target = f"out/{os.path.splitext(os.path.basename(zpath))[0]}" if ('images' in tops or 'labels' in tops) else ('.' if tops=={'out'} else 'out')
        os.makedirs(target, exist_ok=True); zf.extractall(target)
for z in [p for p in sorted(glob.glob(f'{DRIVE_DIR}/*.zip')) if 'dataset' not in os.path.basename(p).lower()]:
    extract_chunk(z); print('extracted', os.path.basename(z))
print('out dirs:', len(glob.glob('out/*/labels')), '| train labels:', len(glob.glob('out/train/labels/*.json')), '| train imgs:', len(glob.glob('out/train/images/*')))

In [ ]:
# ---- labeled CSVs from the JSON labels (img = out/<split>/images/<name>.jpg) ----
import json, glob, os, csv
def build_csv(split):
    rows=[]
    for j in glob.glob(f'out/{split}/labels/*.json'):
        d=json.load(open(j))
        if int(d.get('label',-1))<0: continue
        name=d.get('name', os.path.splitext(os.path.basename(j))[0]); img=f'out/{split}/images/{name}.jpg'
        if os.path.exists(img): rows.append({'path':img,'center':d.get('center',''),'class':'','label':int(d['label'])})
    with open(f'{split}_colab.csv','w',newline='') as f:
        w=csv.DictWriter(f,fieldnames=['path','center','class','label']); w.writeheader(); w.writerows(rows)
    print(f'{split}_colab.csv', len(rows),'pos=',sum(r['label'] for r in rows),'centers=',sorted({r['center'] for r in rows}))
build_csv('train')
try: build_csv('val')
except Exception as e: print('val skip', e)

In [ ]:
# ---- 35-concept target matrix (self-contained: unlabeled out/<dir> + labeled out/train from JSONs) ----
!python -m phase3.build_concept_targets --out phase3/cache/concept_targets.npz

## Stage-1 — concept-supervised pretraining (LIGHT + L2-SP anchor → shape, don't overwrite SSL)

In [ ]:
# light (unfreeze 3, 6 ep) + L2-SP=1.0 keeps SSL features; GRL pushes out center cues.
!python -m phase3.pretrain_concept --targets phase3/cache/concept_targets.npz \
    --unfreeze 3 --epochs 6 --bs 192 --lr 1e-4 --grl 1.0 --l2sp 1.0 --workers 8 --out concept_encoder.pt

## (optional) Gate — did the hardened concept encoder keep/beat SSL features on LOCO PPV@90R?

In [ ]:
!python -m phase3.featurize --csv train_colab.csv --out feats_train_ssl.npz     --workers 4
!python -m phase3.featurize --csv train_colab.csv --out feats_train_concept.npz --workers 4 --backbone concept_encoder.pt
!python -m phase3.concept_gate --ssl feats_train_ssl.npz --concept feats_train_concept.npz --C 1.0

## Stage-2 — downstream from the concept encoder (LOCO check, then 3-seed ensemble ship)

In [ ]:
# honest cross-center check
for hold in ['center_2','center_1']:
    print('================ holdout', hold, '================')
    !python -m phase3.finetune --train-csv train_colab.csv --holdout {hold} --init concept_encoder.pt \
        --unfreeze 6 --epochs 30 --bs 96 --loss bce+rank+pauc --warmup 2 --wise-ft 0.7 --out ft_{hold}.pt

In [ ]:
# ship: train on BOTH centers, 3 seeds -> ensemble
for s in [0,1,2]:
    !python -m phase3.finetune --train-csv train_colab.csv --holdout none --init concept_encoder.pt --seed {s} \
        --unfreeze 6 --epochs 30 --bs 96 --loss bce+rank+pauc --warmup 2 --wise-ft 0.7 --out ship_seed{s}.pt
import shutil
shutil.copy('concept_encoder.pt', f'{DRIVE_DIR}/concept_encoder.pt')
for s in [0,1,2]: shutil.copy(f'ship_seed{s}.pt', f'{DRIVE_DIR}/ship_seed{s}.pt')
print('saved encoder + 3 ensemble members to Drive')

## Inference (offline, per-image, ensemble + hflip-TTA)
```
!python -m phase3.infer --model ship_seed0.pt,ship_seed1.pt,ship_seed2.pt --images-dir <TEST_DIR> --out preds.csv
```
Tune for PPV@90R on the LOCO check: Stage-1 `--l2sp {0.3,1,3}` `--unfreeze {2,3}` `--epochs {4,6}`; Stage-2 `--wise-ft {0.5,0.7,0.9}` `--unfreeze {4,6,8}`. Keep the config with the best **LOCO-worst**.